In [1]:
from infostop import Infostop
import numpy as np
import polars as pl
from pyproj import Transformer

import yaml

In [4]:
with open("../config.yaml") as fp:
    config = yaml.safe_load(fp)
with open("05_detect_stops_config.yaml") as fp:
    stop_detections_config = yaml.safe_load(fp)

In [5]:
# stop detection process
params = stop_detections_config["infostop"]

model = Infostop(
    r1=params["r1"],
    r2=params["r2"],
    min_staying_time=params["min_staying_time"],
    max_time_between=params["max_time_between"],
    min_size=params["min_size"],
    distance_metric=params["distance_metric"],
)

In [6]:
transformer = Transformer.from_crs(4326, config["crs"])

In [7]:
def initial_filter(df: pl.DataFrame, min_pings: int) -> pl.DataFrame:
    """Keep only devices with at least min_pings pings."""
    valid_devices = (
        df.group_by("device_id")
        .len()
        .filter(pl.col("len") >= min_pings)["device_id"]
        .to_list()
    )
    return df.filter(pl.col("device_id").is_in(valid_devices))


In [8]:
data = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/00/pings.parquet"
)


In [9]:
filtered = initial_filter(
    data, min_pings=stop_detections_config["filtering"]["min_pings_per_day"]
)
filtered = filtered.with_columns(
    pl.col("timestamp").dt.epoch(time_unit="s").alias("epoch")
)

In [10]:
# run infostop per device -- data already sorted, no re-sort needed in loop
# catch devices with no stop events (all pings labelled -1)
results = []
for device_id, group in filtered.group_by("device_id"):
    coords = group[["latitude", "longitude", "epoch"]].to_numpy()
    try:
        labels = model.fit_predict(coords)
    except Exception:
        labels = np.full(len(coords), -1)
    group_ = group.with_columns(place_label=labels)

    results.append(group_)

clustered = pl.concat(results)
clustered = clustered.sort(["device_id", "timestamp"]).drop("epoch")

n_stopped = (clustered["place_label"] >= 0).sum()
print(
    f"Stop pings:    {n_stopped:,}  ({n_stopped / len(clustered):.1%} of filtered pings)"
)
print(f"Unique stops:  {clustered['place_label'].clip(lower_bound=0).n_unique():,}")
clustered.head(3)


Stop pings:    3,768,185  (24.4% of filtered pings)
Unique stops:  139


timestamp,device_id,device_aid_type,latitude,longitude,epoch,place_label
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,1727829429,-1
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,1727831869,-1
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,1727832410,-1


In [ ]:
clustered.write_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/clustered_pings/00.parquet"
)

In [12]:
clustered = pl.read_parquet(
    "/mnt/common-hdd/group-dirs/tkom-2026-project/clustered_pings/00.parquet"
)

In [ ]:
# assign stop_id: a sequential integer per device, incrementing each time a new stop begins.
# stop_id is independent of place_label -- the same place visited twice gets two different stop_ids.
# pings with place_label == -1 (movement) get stop_id == -1.

clustered = clustered.with_columns(
    pl.when(pl.col("place_label") >= 0)
    .then(
        (
            (pl.col("place_label") >= 0)
            & (pl.col("place_label").shift(1).over("device_id").fill_null(-1) < 0)
        )
        .cast(pl.Int32)
        .cum_sum()
        .over("device_id")
    )
    .otherwise(-1)
    .alias("stop_id")
)

In [14]:
place_lable_jt = (
    clustered.filter(pl.col.stop_id >= 0)[["device_id", "stop_id", "place_label"]]
    .unique()
    .sort(["device_id", "stop_id"])
)

In [15]:
clustered.filter((pl.col.stop_id == 4) & (pl.col.device_id == 1))

timestamp,device_id,device_aid_type,latitude,longitude,place_label,stop_id
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,i64,i32
2024-10-05 23:53:54 CEST,1,"""AAID""",47.476742,19.075131,8,4
2024-10-06 00:04:27 CEST,1,"""AAID""",47.476742,19.075131,8,4


In [16]:
projected = transformer.transform(clustered["latitude"], clustered["longitude"])
clustered = clustered.with_columns(
    latitude_proj=projected[0], longitude_proj=projected[1]
)

In [17]:
stops = (
    clustered.filter(pl.col.stop_id >= 0)
    .group_by(["device_id", "stop_id"])
    .agg(
        interval_start=pl.col.timestamp.min(),
        interval_stop=pl.col.timestamp.max(),
        interval_lenght=pl.col.timestamp.max() - pl.col.timestamp.min(),
        latitude_mean=pl.col.latitude.mean(),
        longitude_mean=pl.col.longitude.mean(),
        latitude_std=pl.col.latitude_proj.std().round(),
        longitude_std=pl.col.longitude_proj.std().round(),
        number_of_pings=pl.col.longitude.count(),
    )
    .join(place_lable_jt, on=["device_id", "stop_id"])
    .sort(["device_id", "stop_id"])
)
stops.write_parquet("/mnt/common-hdd/group-dirs/tkom-2026-project/stops/00.parquet")
stops

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8
1,4,2024-10-05 23:53:54 CEST,2024-10-06 00:04:27 CEST,10m 33s,47.476742,19.075131,0.0,0.0,2,8
1,5,2024-10-09 04:46:54 CEST,2024-10-09 04:58:36 CEST,11m 42s,47.476778,19.075128,3.0,4.0,2,8
…,…,…,…,…,…,…,…,…,…,…
6584,333,2024-11-17 11:07:57 CET,2024-11-17 11:24:17 CET,16m 20s,47.606873,19.270928,8.0,1.0,3,4
6584,334,2024-11-17 18:15:42 CET,2024-11-17 18:37:33 CET,21m 51s,47.60659,19.270803,14.0,25.0,5,4
6584,335,2024-11-17 21:11:36 CET,2024-11-17 21:26:20 CET,14m 44s,47.606829,19.270937,11.0,6.0,3,4


In [18]:
stops.filter(pl.col.device_id == 1)

device_id,stop_id,interval_start,interval_stop,interval_lenght,latitude_mean,longitude_mean,latitude_std,longitude_std,number_of_pings,place_label
i64,i32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",duration[μs],f64,f64,f64,f64,u32,i64
1,1,2024-10-02 07:24:06 CEST,2024-10-02 07:35:29 CEST,11m 23s,47.661166,19.076356,4.0,0.0,2,17
1,2,2024-10-05 11:06:43 CEST,2024-10-05 11:22:57 CEST,16m 14s,47.498402,19.0404,0.0,0.0,4,14
1,3,2024-10-05 17:49:22 CEST,2024-10-05 17:59:54 CEST,10m 32s,47.476742,19.075131,0.0,0.0,3,8
1,4,2024-10-05 23:53:54 CEST,2024-10-06 00:04:27 CEST,10m 33s,47.476742,19.075131,0.0,0.0,2,8
1,5,2024-10-09 04:46:54 CEST,2024-10-09 04:58:36 CEST,11m 42s,47.476778,19.075128,3.0,4.0,2,8
…,…,…,…,…,…,…,…,…,…,…
1,91,2025-05-18 06:49:07 CEST,2025-05-18 07:12:57 CEST,23m 50s,47.486001,19.062496,5.0,3.0,8,4
1,92,2025-05-18 07:32:47 CEST,2025-05-18 08:05:53 CEST,33m 6s,47.485928,19.06309,67.0,11.0,12,4
1,92,2025-05-18 07:32:47 CEST,2025-05-18 08:05:53 CEST,33m 6s,47.485928,19.06309,67.0,11.0,12,2
